# Yolo - PreTrained

In [ ]:
from ultralytics import YOLO
import os
import json
import shutil

In [ ]:
!pip install ultralytics

## Criar yaml para teste com o respetivo formato para Ultralytics YOLO 

Fazer yaml para os dados 
https://docs.ultralytics.com/pt/datasets/detect/#ultralytics-yolo-format

In [ ]:
json_path = "annotations.json"
output_root = ""

# Criação de pastas
splits = ['train', 'val', 'test']
for split in splits:
    os.makedirs(os.path.join(output_root, "images", split), exist_ok=True)
    os.makedirs(os.path.join(output_root, "labels", split), exist_ok=True)

# Carregar ficheiro JSON
with open(json_path, 'r') as f:
    data = json.load(f)

# Criar índice por id para acesso rápido
id_to_image = {img['id']: img for img in data['images']}

annotations = data['annotations']['pieces']
chess_board = data['annotations']['corners']

# Criação de índice de anotações por imagem_id
annotations_by_image = {}
for ann in annotations:
    image_id = ann['image_id']
    if ann.get('bbox'):
        bbox = ann['bbox']
    else:
        continue
    annotations_by_image.setdefault(image_id, []).append({
    "bbox": bbox,
    "category_id": ann['category_id']
})
    
# Criação de índice de anotações por imagem_id
chess_board_by_image = {}
for ann in chess_board:
    image_id = ann['image_id']
    if ann.get('corners'):
        corners = ann['corners']
    else:
        continue
    chess_board_by_image.setdefault(image_id, []).append({"corners": corners})

# Processar splits
for split in splits:
    ids = data['splits']['chessred2k'].get(split, [])
    for image_id in ids['image_ids']:
        image_info = id_to_image.get(image_id)
        if not image_info:
            print(f"[AVISO] ID {image_id} não encontrado em 'images'")
            continue

        src_path = "chessred2k/" + image_info.get('path')
        if not src_path or not os.path.isfile(src_path):
            print(f"[AVISO] Ficheiro não encontrado: {src_path}")
            continue

        dst_path = os.path.join(output_root, "Images", split, image_info['file_name'])
        shutil.copy2(src_path, dst_path)

        img_width = image_info['width']
        img_height = image_info['height']

        label_path = os.path.join(output_root, "Labels", split, image_info['file_name'].replace(".jpg", ".txt"))

        with open(label_path, 'w') as label_file:
            for items in chess_board_by_image.get(image_id, []):
                corners = items['corners']
                x_br, y_br = corners['bottom_right']
                x_tr, y_tr = corners['top_right']
                x_tf, y_tf = corners['top_left']
                x_bf, y_bf = corners['bottom_left']

                x_min = min(x_br, x_tr, x_tf, x_bf)
                x_max = max(x_br, x_tr, x_tf, x_bf)
                y_min = min(y_br, y_tr, y_tf, y_bf)
                y_max = max(y_br, y_tr, y_tf, y_bf)

                w = x_max - x_min
                h = y_max - y_min
                x_center = x_min + w / 2
                y_center = y_min + h / 2

                # YOLO normalization
                x_center_norm = x_center / img_width
                y_center_norm = y_center / img_height
                w_norm = w / img_width
                h_norm = h / img_height

                label_file.write(f"13 {x_center_norm:.6f} {y_center_norm:.6f} {w_norm:.6f} {h_norm:.6f}\n")
            for items in annotations_by_image.get(image_id, []):
                bbox = items['bbox'] 
                piece_categorie = items['category_id']
                x, y, w, h = bbox

                # Converter para formato YOLO
                x_center = (x + w / 2) / img_width
                y_center = (y + h / 2) / img_height
                w_norm = w / img_width
                h_norm = h / img_height

                label_file.write(f"{piece_categorie} {x_center:.6f} {y_center:.6f} {w_norm:.6f} {h_norm:.6f}\n")

## Train with a pre-trained model

In [ ]:
model = YOLO("yolo11n.yaml")  # build a new model from YAML
model = YOLO("yolo11n.pt")  # load a pretrained model
model = YOLO("yolo11n.yaml").load("yolo11n.pt")  # build from YAML and transfer weights

In [ ]:
model.train(data="data.yaml", epochs=150, imgsz=640, patience=5)

In [ ]:
# Load a model
model = YOLO("path/to/best.pt")  # load a custom model

# Validate the model
metrics = model.val()  # no arguments needed, dataset and settings remembered
metrics.box.map  # map50-95
metrics.box.map50  # map50
metrics.box.map75  # map75
metrics.box.maps

# Teste the model
metrics = model.val(split="test")  
metrics.box.map  # map50-95
metrics.box.map50  # map50
metrics.box.map75  # map75
metrics.box.maps

In [ ]:
# Predict with the model
results = model("image.jpg")  # predict on an image

# Access the results
for result in results:
    xywh = result.boxes.xywh  # center-x, center-y, width, height
    xywhn = result.boxes.xywhn  # normalized
    xyxy = result.boxes.xyxy  # top-left-x, top-left-y, bottom-right-x, bottom-right-y
    xyxyn = result.boxes.xyxyn  # normalized
    names = [result.names[cls.item()] for cls in result.boxes.cls.int()]  # class name of each box
    confs = result.boxes.conf  # confidence score of each box